Object Detection using YOLO

Classes :  5
selected classes = car, airplane, truck, motorcycle, bus

In [1]:
pip install ultralytics pycocotools scikit-learn pillow matplotlib


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached ultralytics-8.4.24-py3-none-any.whl (1.2 MB)
     -------------------------------------- 80.2/80.2 kB 344.5 kB/s eta 0:00:00
  Using cached matplotlib-3.10.8-cp310-cp310-win_amd64.whl (8.1 MB)
  Using cached polars-1.39.3-py3-none-any.whl (823 kB)
  Using cached torch-2.10.0-cp310-cp310-win_amd64.whl (113.7 MB)
  Using cached torchvision-0.25.0-cp310-cp310-win_amd64.whl (3.7 MB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl (28 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
  Using cached kiwisolver-1.5.0-cp310-cp310-win_amd64.whl (73 kB)
  Using cached fonttools-4.62.1-cp310-cp310-win_amd64.whl (1.6 MB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl (221 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
  Using cached filelock-3.25.2-py3-none-any.whl (26 kB)
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)


In [9]:
from pycocotools.coco import COCO
import os, shutil, json
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ANNOTATIONS_FILE = 'annotations_trainval2017/annotations/instances_val2017.json'
IMAGES_DIR = 'val2017/val2017'    
DATASET_DIR = 'dataset'
CLASSES = ['car', 'airplane', 'truck', 'motorcycle', 'bus'] 



In [10]:
coco = COCO(ANNOTATIONS_FILE)

cat_ids = coco.getCatIds(catNms=CLASSES)
cats = coco.loadCats(cat_ids)
cat_id_to_idx = {cat['id']: i for i, cat in enumerate(cats)}

print("classes and their COCO IDs:")
for cat in cats:
    print(f"  {cat['name']} → COCO ID: {cat['id']} → YOLO ID: {cat_id_to_idx[cat['id']]}")

img_ids = set()
for cat_id in cat_ids:
    img_ids.update(coco.getImgIds(catIds=[cat_id]))
img_ids = list(img_ids)
print(f"\nTotal images found: {len(img_ids)}")

loading annotations into memory...
Done (t=1.81s)
creating index...
index created!
classes and their COCO IDs:
  car → COCO ID: 3 → YOLO ID: 0
  motorcycle → COCO ID: 4 → YOLO ID: 1
  airplane → COCO ID: 5 → YOLO ID: 2
  bus → COCO ID: 6 → YOLO ID: 3
  truck → COCO ID: 8 → YOLO ID: 4

Total images found: 869


In [11]:
for split in ['train', 'val']:
    os.makedirs(f'{DATASET_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{DATASET_DIR}/labels/{split}', exist_ok=True)

print("Folders created:")
print("  dataset/images/train/")
print("  dataset/images/val/")
print("  dataset/labels/train/")
print("  dataset/labels/val/")

Folders created:
  dataset/images/train/
  dataset/images/val/
  dataset/labels/train/
  dataset/labels/val/


In [12]:
def convert_to_yolo(bbox, img_w, img_h):    # coco to yolo format [x,y,w,h] -> [cx,cy,w,h]
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    w  = w / img_w
    h  = h / img_h
    return cx, cy, w, h

train_ids, val_ids = train_test_split(img_ids, test_size=0.2, random_state=42)
print(f"Train images: {len(train_ids)}")
print(f"Val images:   {len(val_ids)}")

def process_split(img_ids_list, split_name):
    skipped = 0
    for img_id in img_ids_list:
        img_info = coco.loadImgs(img_id)[0]
        file_name = img_info['file_name']
        img_w = img_info['width']
        img_h = img_info['height']

        ann_ids = coco.getAnnIds(imgIds=img_id, catIds=cat_ids, iscrowd=False)
        anns = coco.loadAnns(ann_ids)

        if len(anns) == 0:
            skipped += 1
            continue

        src = os.path.join(IMAGES_DIR, file_name)
        dst = os.path.join(DATASET_DIR, 'images', split_name, file_name)
        if os.path.exists(src):
            shutil.copy(src, dst)

        label_path = os.path.join(DATASET_DIR, 'labels', split_name,
                                  file_name.replace('.jpg', '.txt'))
        with open(label_path, 'w') as f:
            for ann in anns:
                if ann['category_id'] not in cat_id_to_idx:
                    continue
                class_idx = cat_id_to_idx[ann['category_id']]
                cx, cy, w, h = convert_to_yolo(ann['bbox'], img_w, img_h)
                f.write(f"{class_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

    print(f"  Done {split_name}: {len(img_ids_list) - skipped} images processed, {skipped} skipped")

print("\nProcessing train set...")
process_split(train_ids, 'train')
print("Processing val set...")
process_split(val_ids, 'val')
print("\nDataset ready!")

Train images: 695
Val images:   174

Processing train set...
  Done train: 695 images processed, 0 skipped
Processing val set...
  Done val: 174 images processed, 0 skipped

Dataset ready!


In [13]:
yaml_content = f"""path: {os.path.abspath(DATASET_DIR)}
train: images/train
val: images/val

nc: 5
names: {CLASSES}
"""

with open('coco_subset.yaml', 'w') as f:
    f.write(yaml_content)

print("coco_subset.yaml created:")
print(yaml_content)

coco_subset.yaml created:
path: C:\Users\User\Desktop\Dlp Assignment2 (23k-0652)\dataset
train: images/train
val: images/val

nc: 5
names: ['car', 'airplane', 'truck', 'motorcycle', 'bus']



In [14]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='coco_subset.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    name='coco_subset_run',
    pretrained=True
)

print("Training complete!")
print("Best weights saved at: runs/detect/coco_subset_run/weights/best.pt")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\User\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n.pt... <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)>
Ultralytics 8.4.24  Python-3.10.10 torch-2.10.0+cpu CPU (Intel Core i5-6200U 2.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco_subset.yaml, degrees=0.0, deterministic=True, device=cp

In [15]:
trained_model = YOLO('runs/detect/coco_subset_run/weights/best.pt')

metrics = trained_model.val(data='coco_subset.yaml')

print("\n\tRESULTS FOR REPORT")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print(f"mAP@0.5:    {metrics.box.map50:.4f}")

Ultralytics 8.4.24  Python-3.10.10 torch-2.10.0+cpu CPU (Intel Core i5-6200U 2.30GHz)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 616.1184.3 MB/s, size: 191.0 KB)
val: Scanning C:\Users\User\Desktop\Dlp Assignment2 (23k-0652)\dataset\labels\val.cache... 174 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 174/174  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.7s/it 29.5s2.9s
                   all        174        620      0.663      0.456      0.514       0.34
                   car         99        357       0.67      0.466      0.539      0.316
              airplane         35        102      0.673      0.344      0.407      0.221
                 truck         20         30      0.799      0.529      0.606      0.407
            motorcycle         41         52      0.732      0.635      0.689       0.54
               

In [16]:
import glob, random

val_images = glob.glob(f'{DATASET_DIR}/images/val/*.jpg')
sample_images = random.sample(val_images, 10)

trained_model = YOLO('runs/detect/coco_subset_run/weights/best.pt')

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
axes = axes.flatten()

for i, img_path in enumerate(sample_images):
    results = trained_model(img_path, verbose=False)[0]
    
    img = Image.open(img_path)
    ax = axes[i]
    ax.imshow(img)
    
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        label = f"{CLASSES[cls]} {conf:.2f}"
        
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, label, color='red', fontsize=8,
                bbox=dict(facecolor='white', alpha=0.5))
    
    ax.axis('off')
    ax.set_title(f"Image {i+1}")

plt.tight_layout()
plt.savefig('detection_outputs.png', dpi=150)
plt.show()
print("Saved as detection_outputs.png")

<Figure size 2500x1000 with 10 Axes>

Saved as detection_outputs.png
